### Part 2: Stability Experiment

### 1. System alignment

In [ ]:
#Importing necessary libraries
import pandas as pd
import time
import sys
import yaml
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.llm_client import LLMClient
from utils.prompts import render

### 2. Initialization

In [ ]:
# Load config to dynamically select the correct provider 
with open(project_root / "config" / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Initialize client using the mission-recommended 'reason' technique
client = LLMClient(provider=config['providers']['default'], technique="reason")

# Define file paths
scenario_path = project_root / "data" / "scenarios.txt"
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "stability_experiment_results.csv"

# Load scenarios: split by double newline to maintain scenario blocks
if scenario_path.exists():
    content = scenario_path.read_text(encoding="utf-8")
    scenarios = [s.strip() for s in content.split("\n\n") if s.strip()]
    print(f"✅ Loaded {len(scenarios)} scenarios.")
else:
    print("❌ Error: data/scenarios.txt not found.")
    scenarios = []

experiment_logs = []

### 3. Execution Loop (Safe vs Chaos)

In [ ]:
for scenario in scenarios:
    print(f"\n🚀 Stress Testing Scenario: {scenario[:50]}...")
    
    # Render using the Chain-of-Thought reasoning template
    prompt_text, _ = render(
        "cot_reasoning.v1", 
        role="Crisis Intelligence Analyst", 
        problem=scenario
    )
    
    # A. SAFE MODE: 1 Run at Temperature 0.0 
    # We pass temperature explicitly to override any class defaults
    res_safe = client.chat(
        [{"role": "user", "content": prompt_text}], 
        temperature=0.0
    )
    experiment_logs.append({
        "Scenario": scenario, 
        "Mode": "Safe (0.0)", 
        "Output": res_safe['text'].strip()
    })
    
    # B. CHAOS MODE: 3 Runs at Temperature 1.0
    for i in range(3):
        res_chaos = client.chat(
            [{"role": "user", "content": prompt_text}], 
            temperature=1.0 
        )
        experiment_logs.append({
            "Scenario": scenario, 
            "Mode": f"Chaos (1.0) - Run {i+1}", 
            "Output": res_chaos['text'].strip()
        })
        time.sleep(1) # Safety interval for API rate limits 

### 4. Visualization and Output

In [ ]:
df_exp = pd.DataFrame(experiment_logs)

for scenario in scenarios:
    print(f"\n{'='*80}\nSCENARIO: {scenario}\n{'='*80}")
    scenario_data = df_exp[df_exp['Scenario'] == scenario]
    
    for _, row in scenario_data.iterrows():
        print(f"\n>>> MODE: {row['Mode']}")
        print(f"{row['Output']}")
        print("-" * 40)

### 5. Deliverable

Semantic & Inference Drift (Scenario A):

Safe Mode: Consistently concludes that the district is not explicitly in the text (stating it's not given).

Chaos Run 3: Drifts into inference, explicitly claiming the location is in "Kandy District" based on the title, which violates the strict extraction constraint seen in Safe Mode.

Formatting Drift (Scenario B):

Safe Mode: Follows the prompt instructions for step-by-step numbers strictly.

Chaos Run 2: Becomes conversational, adding an intro: "To address the given crisis scenario, let's break it down step by step...". This is a classic example of high-temperature "verbosity drift."

Hallucination/Inconsistency:

Chaos runs that cut off mid-sentence (like Chaos Run 1 for Scenario A) or invents specific details about the insulin needs or battery types that weren't in the source text.